# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1
# YOUR CODE HERE

import plotly.express as px

# 1. Load Gapminder dataset
gm = px.data.gapminder()

# 2. Filter data for 2007
gm_2007 = gm[gm['year'] == 2007].copy()

# 3. Calculate global average life expectancy
global_mean = gm_2007['lifeExp'].mean()

# 4. Calculate deviation from global average
gm_2007['lifeExp_deviation'] = gm_2007['lifeExp'] - global_mean

# 5. Identify the region furthest below average
region_deviation = (
    gm_2007.groupby('continent')['lifeExp_deviation']
    .mean()
    .sort_values()
)

lowest_region = region_deviation.index[0]

# 6. Create choropleth map
fig = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    color='lifeExp_deviation',
    hover_name='country',
    hover_data={
        'lifeExp': ':.2f',
        'lifeExp_deviation': ':+.2f',
        'iso_alpha': False
    },
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title=(
        f"Life Expectancy Relative to Global Average (2007)"
        f"<br><sup>Region furthest below average: {lowest_region}</sup>"
    ),
    labels={
        'lifeExp': 'Life Expectancy (years)',
        'lifeExp_deviation': 'Deviation from Global Average (years)'
    }
)

# 7. Improve layout
fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    coloraxis_colorbar=dict(
        title='Years above/below<br>global average'
    ),
    title_x=0.5
)

fig.show()



## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** *(URL or description)*

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox`) **and reason:**

*Your answer here*

**Colour scale chosen** (sequential or diverging) **and reason:**

*Your answer here*


In [ ]:
# Task 2
# YOUR CODE HERE

# Load your GeoJSON
# with open('your_file.geojson', 'r') as file:
#     my_geojson = json.load(file)

# Inspect the properties to find the right featureidkey
# print(my_geojson['features'][0]['properties'])



## Task 2 – World GDP per Capita Choropleth (2007)

**GeoJSON source:** GitHub – datasets/geo-countries
https://github.com/datasets/geo-countries

**Visualization choice:** I selected `px.choropleth()` because it provides a straightforward way to visualize country-level geographical data without requiring an external map tile provider.

**Feature ID key:** `properties.ISO3166-1-Alpha-3`. This property contains the three-letter ISO country code, which matches the `iso_alpha` column in the Gapminder dataset.

**Colour scale:** I selected a sequential colour scale because GDP per capita is a continuous, non-negative variable. A sequential scale makes it easier to distinguish countries with lower and higher GDP per capita.

**Research insight:** The visualization explores the differences in GDP per capita across countries in 2007 and identifies the country with the highest GDP per capita.


In [ ]:

import json
import requests

# Download GeoJSON from GitHub
geojson_url = (
    "https://raw.githubusercontent.com/"
    "datasets/geo-countries/main/data/countries.geojson"
)

response = requests.get(geojson_url)
response.raise_for_status()

my_geojson = response.json()

# Inspect the first feature
print(my_geojson['features'][0]['properties'])


In [ ]:

import plotly.express as px
import pandas as pd

# Load Gapminder dataset
gm = px.data.gapminder()

# Filter for 2007
gm_2007 = gm[gm['year'] == 2007].copy()

# Select required columns
gdp_data = gm_2007[
    ['country', 'iso_alpha', 'gdpPercap']
].copy()

# Rename columns
gdp_data = gdp_data.rename(columns={
    'iso_alpha': 'country_code',
    'gdpPercap': 'gdp_per_capita'
})

# Check the dataset
print(gdp_data.head())


In [ ]:

# Find country with the highest GDP per capita
highest_country = gdp_data.loc[
    gdp_data['gdp_per_capita'].idxmax(),
    'country'
]

highest_gdp = gdp_data['gdp_per_capita'].max()

# Create choropleth
fig = px.choropleth(
    gdp_data,
    geojson=my_geojson,
    locations='country_code',
    featureidkey='properties.ISO3166-1-Alpha-3',
    color='gdp_per_capita',
    hover_name='country',
    hover_data={
        'country_code': False,
        'gdp_per_capita': ':.2f'
    },
    color_continuous_scale='YlOrRd',
    title=(
        f'GDP per Capita Across the World (2007)'
        f'<br><sup>Highest GDP per capita: {highest_country}</sup>'
    ),
    labels={
        'gdp_per_capita': 'GDP per Capita (USD)'
    }
)

# Improve map layout
fig.update_geos(
    showcoastlines=True,
    showframe=False,
    projection_type='natural earth'
)

fig.update_layout(
    title_x=0.5,
    coloraxis_colorbar=dict(
        title='GDP per Capita<br>(USD)'
    )
)

fig.show()


In [ ]:
ls -lh *.ipynb